Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

<link rel="stylesheet" href="/mediapipe/site.css">

# Hand gesture recognition model customization guide

<table align="left" class="buttons">
  <td>
    <a href="https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/colab-logo-32px_1920.png" alt="Colab logo"> Run in Colab
    </a>
  </td>

  <td>
    <a href="https://github.com/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/github-logo-32px_1920.png" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
</table>

In [1]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

The MediaPipe Model Maker package is a low-code solution for customizing on-device machine learning (ML) Models.

This notebook shows the end-to-end process of customizing a gesture recognizer model for recognizing some common hand gestures in the [HaGRID](https://www.kaggle.com/datasets/innominate817/hagrid-sample-30k-384p) dataset.

## Prerequisites

Install the MediaPipe Model Maker package.

In [2]:
# !pip install --upgrade pip
# !pip install mediapipe-model-maker

Import the required libraries.

In [3]:
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt

2026-03-30 22:53:03.909656: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-30 22:53:03.930667: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-30 22:53:03.930695: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-30 22:53:03.931358: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-30 22:53:03.934952: I tensorflow/core/platform/cpu_feature_guar

## Simple End-to-End Example

This end-to-end example uses Model Maker to customize a model for on-device gesture recognition.

### Get the dataset

The dataset for gesture recognition in model maker requires the following format: `<dataset_path>/<label_name>/<img_name>.*`. In addition, one of the label names (`label_names`) must be `none`. The `none` label represents any gesture that isn't classified as one of the other gestures.

This example uses a rock paper scissors dataset sample which is downloaded from GCS.

In [16]:
# !wget https://storage.googleapis.com/mediapipe-tasks/gesture_recognizer/rps_data_sample.zip
# !unzip rps_data_sample.zip
dataset_path = "/media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped"
export_dir = "/media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data/test_own_datasets/v2"

In [6]:
!python3 erase_hand.py --images-dir /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data --label palm

2026-03-30 22:55:44.600117: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-30 22:55:44.600149: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-30 22:55:44.600700: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-30 22:55:45.132185: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Saving all cropped images to: /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/palm
Processing directory: /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data/raw_hands_1/palm
I0000 00:00:1774900545.775319 

In [7]:
!python3 erase_hand.py --images-dir /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data --label rock

2026-03-30 23:07:49.617920: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-30 23:07:49.617952: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-30 23:07:49.618485: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-30 23:07:50.130870: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Saving all cropped images to: /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/rock
Processing directory: /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data/raw_hands_1/palm
I0000 00:00:1774901270.797641 

In [8]:
!python3 erase_hand.py --images-dir /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data --label none

2026-03-30 23:11:42.913327: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-30 23:11:42.913354: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-30 23:11:42.913879: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-30 23:11:43.435318: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Saving all cropped images to: /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/none
Processing directory: /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data/raw_hands_1/palm
I0000 00:00:1774901504.075177 

Verify the rock paper scissors dataset by printing the labels. There should be 4 gesture labels, with one of them being the `none` gesture.

In [17]:
print(dataset_path)
labels = []
for i in os.listdir(dataset_path):
  if os.path.isdir(os.path.join(dataset_path, i)):
    labels.append(i)
print(labels)

/media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped
['none', 'rock', 'palm']


To better understand the dataset, plot a couple of example images for each gesture.

### Run the example
The workflow consists of 4 steps which have been separated into their own code blocks.

**Load the dataset**

Load the dataset located at `dataset_path` by using the `Dataset.from_folder` method. When loading the dataset, run the pre-packaged hand detection model from MediaPipe Hands to detect the hand landmarks from the images. Any images without detected hands are ommitted from the dataset. The resulting dataset will contain the extracted hand landmark positions from each image, rather than images themselves.

The `HandDataPreprocessingParams` class contains two configurable options for the data loading process:
* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection.

Split the dataset: 80% for training, 10% for validation, and 10% for testing.

In [18]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/none/PXL_20260322_215559878.jpg
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/rock/2026-03-27 18-36-26.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/rock/2026-03-23 20-18-43_1774286454_21256698.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/palm/2026-03-27 19-05-46_1774627779_79452825.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/palm/2026-03-23 20-08-12.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/none/2026-03-27 17-33-49.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/none/2026-03-27 19-06-16.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T

I0000 00:00:1774905810.453044 3370375 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1774905810.540357 3476085 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.126.16), renderer: NVIDIA GeForce RTX 4090/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1774905810.551794 3476090 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774905810.562895 3476105 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/home/imit-learn/anaconda3/envs/hoverAI_2/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype(

INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/palm/2026-03-23 20-31-25_1774287099.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/none/2026-03-27 17-25-43.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/palm/2026-03-23 19-22-58_62188741.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/rock/PXL_20260322_215050985.jpg
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/palm/PXL_20260322_215046101.jpg
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/none/2026-03-23 19-23-52_1774283259.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/cropped/rock/2026-03-27 18-37-27.JPG
INFO:tensorflow:Loading image /media/imit-learn/ISR_2T3/Hove

2026-03-31 00:26:44.881200: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-31 00:26:45.038061: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


INFO:tensorflow:Load valid hands with size: 4801, num_label: 3, labels: none,palm,rock.


INFO:tensorflow:Load valid hands with size: 4801, num_label: 3, labels: none,palm,rock.


**Train the model**

Train the custom gesture recognizer by using the create method and passing in the training data, validation data, model options, and hyperparameters. For more information on model options and hyperparameters, see the [Hyperparameters](#hyperparameters) section below.

In [21]:
hparams = gesture_recognizer.HParams(export_dir=export_dir, epochs=20)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)




Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization_1 (Bat  (None, 128)               512       
 chNormalization)                                                
                                                                 
 re_lu_1 (ReLU)              (None, 128)               0         
                                                                 
 dropout_1 (Dropout)         (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 3)                 387       
 out (Dense)                                                     
                                                           

INFO:tensorflow:Training the models...


Resuming from /media/imit-learn/ISR_2T3/HoverAI_2/gesture_real_time_control/data/test_own_datasets/v2/epoch_models/model-0010
Epoch 1/20
1920/1920 [==============================] - 2s 761us/step - loss: 0.4875 - categorical_accuracy: 0.3729 - val_loss: 0.5340 - val_categorical_accuracy: 0.3104 - lr: 0.0010
Epoch 2/20
1920/1920 [==============================] - 1s 715us/step - loss: 0.4866 - categorical_accuracy: 0.3784 - val_loss: 0.5322 - val_categorical_accuracy: 0.3104 - lr: 9.9000e-04
Epoch 3/20
1920/1920 [==============================] - 1s 749us/step - loss: 0.4864 - categorical_accuracy: 0.3784 - val_loss: 0.5359 - val_categorical_accuracy: 0.3187 - lr: 9.8010e-04
Epoch 4/20
1920/1920 [==============================] - 1s 686us/step - loss: 0.4849 - categorical_accuracy: 0.3784 - val_loss: 0.5364 - val_categorical_accuracy: 0.3313 - lr: 9.7030e-04
Epoch 5/20
1920/1920 [==============================] - 1s 704us/step - loss: 0.4853 - categorical_accuracy: 0.3747 - val_loss: 0.

**Evaluate the model performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

In [22]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

481/481 [==============================] - 1s 318us/step - loss: 0.5517 - categorical_accuracy: 0.2994
Test loss:0.5517495274543762, Test accuracy:0.29937630891799927


**Export to Tensorflow Lite Model**

After creating the model, convert and export it to a Tensorflow Lite model format for later use on an on-device application. The export also includes model metadata, which includes the label file.

In [24]:
model.export_model()
!ls exported_model

Using existing files at /tmp/model_maker/gesture_recognizer/gesture_embedder.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/hand_landmark_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/canned_gesture_classifier.tflite
INFO:tensorflow:Assets written to: /tmp/tmpsrqk3gt0/saved_model/assets


INFO:tensorflow:Assets written to: /tmp/tmpsrqk3gt0/saved_model/assets
2026-03-31 00:29:54.368066: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-03-31 00:29:54.368084: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-03-31 00:29:54.368175: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpsrqk3gt0/saved_model
2026-03-31 00:29:54.368672: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-03-31 00:29:54.368677: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpsrqk3gt0/saved_model
2026-03-31 00:29:54.369749: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-03-31 00:29:54.379970: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpsrqk3gt0/saved_model
2026-03-31 00:29:54.383486: I ten

best_model_weights.data-00000-of-00001	epoch_models		 metadata.json
best_model_weights.index		gesture_recognizer.task
checkpoint				logs
